# IEEE 8500-node feeder — OpenDSS timing (GNN2)

**This notebook file:** `IEEE8500_OpenDSS_timing.ipynb` (in your `GNN2` project folder)

**Not** the `8500-node` **folder** — that folder only holds OpenDSS `.dss` data files.

**Goal:** find `Master.dss`, run snapshot power flow, and measure **median `Solve()` time (ms)** to compare later with GNN inference at batch size 1.

| Step | What |
|------|------|
| **1** | Point OpenDSS at `Master.dss` |
| **2** | Load circuit + time 50 snapshot solves |
| **3** | Generate load-type CSVs (`run_loadtype_dataset_8500`) |
| **4** | Build `edge_index` / `edge_attr` (`build_graph_8500`) |
| **5** | Stack node `X` / `Y` tensors (`assemble_dataset_tensors_8500`) |
| **6** | MLP sweep: 5 architectures, best saved (`train_mlp_8500`) |

Run cells **in order** (top to bottom).


## Step 1 — Path to `Master.dss`

The feeder is a **folder of `.dss` files**. The **entry file** is **`Master.dss`**. Your copy lives in **`GNN2/8500-node`**.

**This step:** set `LOCAL_8500_DIR` if you move the folder, run the code cell, and confirm it prints **`Master.dss`**.

**Pass:** `Master.dss` exists at the printed path.


In [1]:
import pathlib
import opendssdirect

# --- edit only if you move the feeder folder ---
LOCAL_8500_DIR = pathlib.Path(r"C:\Users\alita\OneDrive\Desktop\GNN2\8500-node")
LOCAL_8500_MASTER = LOCAL_8500_DIR / "Master.dss"

DSS_8500: pathlib.Path | None = None

if LOCAL_8500_MASTER.is_file():
    DSS_8500 = LOCAL_8500_MASTER
else:
    for base in (
        pathlib.Path(opendssdirect.__file__).parent,
        pathlib.Path(r"C:\Program Files\OpenDSS"),
        pathlib.Path(r"C:\OpenDSS"),
    ):
        if not base.exists():
            continue
        try:
            for f in base.rglob("Master.dss"):
                if "8500" in str(f).lower():
                    DSS_8500 = f
                    break
        except (OSError, PermissionError):
            pass
        if DSS_8500 is not None:
            break

if DSS_8500 is None:
    print("Master.dss not found. Place the full 8500-node folder at:")
    print(" ", LOCAL_8500_DIR)
else:
    print("Using entry file:", DSS_8500.resolve())


Using entry file: C:\Users\alita\OneDrive\Desktop\GNN2\8500-node\Master.dss


**Checkpoint — Step 1 done**

You have the OpenDSS entry file **`Master.dss`**.

**Next:** Step 2 loads the circuit and measures solve time.


## Step 2 — Load circuit and profile `Solve()`

**This step:** `Redirect` `Master.dss`, run one solve (convergence + bus counts), then **50** snapshot solves and report **median / mean / min / max** in **ms**.

**Note:** `from opendssdirect import dss` is required — the **package** is not callable (`import opendssdirect as dss` breaks `dss(...)`).

**Pass:** `Converged: True`, buses ~4876, node-phases ~8500+, and a stable median time.


In [2]:
# Step 2 — load + time snapshot solves
import statistics
import time

import pathlib

from opendssdirect import dss

LOCAL_8500_MASTER = pathlib.Path(r"C:\Users\alita\OneDrive\Desktop\GNN2\8500-node\Master.dss")
_d = globals().get("DSS_8500")
if _d is not None:
    DSS_8500 = _d
elif LOCAL_8500_MASTER.is_file():
    DSS_8500 = LOCAL_8500_MASTER
else:
    DSS_8500 = None
if DSS_8500 is None:
    raise RuntimeError("Run Step 1 or place Master.dss at GNN2/8500-node/Master.dss")

dss(f'redirect "{DSS_8500.resolve()}"')
dss.Solution.Mode(1)
dss.Solution.Solve()

converged = dss.Solution.Converged()
n_buses = len(dss.Circuit.AllBusNames())
all_nodes = dss.Circuit.AllNodeNames()
n_nodes = len([n for n in all_nodes if "." in n and n.split(".")[1] in ("1", "2", "3")])

print("Circuit:", DSS_8500)
print(f"  Converged    : {converged}")
print(f"  Buses        : {n_buses}")
print(f"  Node-phases  : {n_nodes} (phases 1/2/3)")
print(f"  Control iter : {dss.Solution.ControlIterations()}")
print(f"  PF iter      : {dss.Solution.Iterations()}")

times_ms = []
for _ in range(50):
    t0 = time.perf_counter()
    dss.Solution.Solve()
    times_ms.append((time.perf_counter() - t0) * 1000.0)

med = statistics.median(times_ms)
print()
print("Solve() time over 50 runs (ms):")
print(f"  median : {med:.3f} ms")
print(f"  mean   : {statistics.mean(times_ms):.3f} ms")
print(f"  min    : {min(times_ms):.3f} ms")
print(f"  max    : {max(times_ms):.3f} ms")
print()
print(f"→ Rough batch-1 target vs pure Solve(): beat ~{med:.1f} ms/step")


Circuit: C:\Users\alita\OneDrive\Desktop\GNN2\8500-node\Master.dss
  Converged    : True
  Buses        : 4876
  Node-phases  : 8541 (phases 1/2/3)
  Control iter : 1
  PF iter      : 2

Solve() time over 50 runs (ms):
  median : 47.016 ms
  mean   : 49.884 ms
  min    : 42.119 ms
  max    : 80.902 ms

→ Rough batch-1 target vs pure Solve(): beat ~47.0 ms/step


**Checkpoint — Step 2 done**

You now have **median `Solve()` time** for this machine on the 8500-node case. Use it as an order-of-magnitude bar when comparing to a surrogate (plus PyTorch overhead, etc.).

---

## Step 3 — Generate load-type dataset (8500)

Run the **next code cell** to execute `run_loadtype_dataset_8500.py`. It writes the same style of CSVs as `run_loadtype_dataset.py`, but for **`8500-node/Master.dss`**, under **`datasets_gnn2/loadtype_8500/`**.

Edit **`n_samples`** (and seeds) in that cell if you want a smaller test first.

---

## Step 4 — Build static graph tensors (`edge_index`, `edge_attr`)

Run **after Step 3** so `gnn_edges_phase_static.csv` and `gnn_node_index_master.csv` exist. The script **`build_graph_8500.py`** writes **`graph_tensors/edge_index.pt`**, **`edge_attr.pt`**, **`graph_meta.json`**, and **`node_index_map.json`** under **`datasets_gnn2/loadtype_8500/`**.

---

## Step 5 — Stack node features / targets (`X`, `Y`)

Run **after Steps 3–4** so `gnn_node_features_and_targets.csv` exists and **`graph_meta.json`** can validate **N**. **`assemble_dataset_tensors_8500.py`** writes **`dataset_tensors/X.pt`**, **`Y.pt`**, optional **`Y_angle.pt`** (`vang_deg`), and **`tensor_manifest.json`**.

---

## Step 6 — MLP baseline (checklist *Step 5*)

Run **after Step 5** so **`X.pt`** / **`Y.pt`** exist. **`train_mlp_8500.py`** trains **five** MLP shapes on **MSE(|V|) only** (no angle), saves each under **`mlp_sweep_8500/<name>/`**, writes **`sweep_summary.json`**, and copies the **best** test MAE to **`mlp_sweep_8500/best/`**. For a **single** run use `train_mlp_baseline_8500()`. The checklist target is **test MAE under 0.005 pu** on \|V\| — expect to **increase `n_samples`** (e.g. 5k–10k) in Step 3 first.

---

## Later (outside this notebook)

- Train **`train_gnn_8500`** (checklist Step 6), then benchmark inference vs OpenDSS.


In [ ]:
# Step 3 — same pattern as GNN2 notebook: chdir to repo, run dataset script
import os

try:
    import run_loadtype_dataset_8500 as _lt8500

    os.chdir(os.path.dirname(os.path.abspath(_lt8500.__file__)))
except Exception:
    pass

# Same as: exec(open("run_loadtype_dataset_8500.py", encoding="utf-8").read())
from run_loadtype_dataset_8500 import generate_gnn_snapshot_dataset_loadtype_8500

generate_gnn_snapshot_dataset_loadtype_8500(
    n_samples=500,
    master_seed=20260322,
    sigma_load=0.12,
    sigma_pv=0.12,
)

## Step 4 — Static graph from dataset CSVs

**Pass:** `build_graph_8500.build_static_graph_8500()` prints shapes and writes files under `datasets_gnn2/loadtype_8500/graph_tensors/`.

**Next:** Step 5 — stack **`X`** / **`Y`** tensors from `gnn_node_features_and_targets.csv`.

In [ ]:
# Step 4 — edge_index / edge_attr .pt files (requires Step 3 CSVs)
import os

try:
    import build_graph_8500 as _bg
    os.chdir(os.path.dirname(os.path.abspath(_bg.__file__)))
except Exception:
    pass

from build_graph_8500 import build_static_graph_8500

build_static_graph_8500()

## Step 5 — Stacked tensors for training

**Pass:** `assemble_dataset_tensors_8500.assemble_dataset_tensors_8500()` prints shapes and writes **`dataset_tensors/X.pt`**, **`Y.pt`**, **`tensor_manifest.json`**.

Uses the same **14** load-type input columns as `run_gnn3_best7_train.LOADTYPE_FEAT`; target defaults to **`vmag_pu`**, with **`Y_angle.pt`** from **`vang_deg`** when present.

**Next:** Step 6 — five-MLP sweep (`train_mlp_architecture_sweep_8500`).

In [5]:
# Step 5 — X [S,N,F], Y [S,N] aligned with graph node order (needs Steps 3–4)
import os

try:
    import assemble_dataset_tensors_8500 as _ad
    os.chdir(os.path.dirname(os.path.abspath(_ad.__file__)))
except Exception:
    pass

from assemble_dataset_tensors_8500 import assemble_dataset_tensors_8500

assemble_dataset_tensors_8500()

[assemble_dataset_tensors_8500] Saved to C:\Users\alita\OneDrive\Desktop\GNN2\datasets_gnn2\loadtype_8500\dataset_tensors/
  X (500, 8541, 14)  Y (500, 8541)  target='vmag_pu'
  samples=500 (dropped incomplete samples vs N=8541)


{'dataset_dir': 'C:\\Users\\alita\\OneDrive\\Desktop\\GNN2\\datasets_gnn2\\loadtype_8500',
 'num_samples': 500,
 'num_nodes': 8541,
 'num_features': 14,
 'feature_columns': ['electrical_distance_ohm',
  'm1_p_kw',
  'm1_q_kvar',
  'm2_p_kw',
  'm2_q_kvar',
  'm4_p_kw',
  'm4_q_kvar',
  'm5_p_kw',
  'm5_q_kvar',
  'q_cap_kvar',
  'p_pv_kw',
  'q_pv_kvar',
  'p_sys_balance_kw',
  'q_sys_balance_kvar'],
 'target_column': 'vmag_pu',
 'X_path': 'C:\\Users\\alita\\OneDrive\\Desktop\\GNN2\\datasets_gnn2\\loadtype_8500\\dataset_tensors\\X.pt',
 'Y_path': 'C:\\Users\\alita\\OneDrive\\Desktop\\GNN2\\datasets_gnn2\\loadtype_8500\\dataset_tensors\\Y.pt',
 'X_shape': [500, 8541, 14],
 'Y_shape': [500, 8541],
 'sample_ids': [0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38,
  39,
  40,
  41,
  42,
  43,
  44,
  45,
  46,
  47,
  4

## Step 6 — MLP sweep (LaTeX checklist Step 5)

**Pass:** `train_mlp_architecture_sweep_8500()` runs **five** architectures (**|V| MSE only**), saves each under **`mlp_sweep_8500/<name>/`**, **`sweep_summary.json`**, and **`best/`** (lowest test MAE on \|V\|).

Default **100** epochs × 5 runs is slow on CPU; reduce **`epochs`** for a smoke test.

In [ ]:
# Step 6 — five MLP architectures; loss = MSE(|V|) only; best -> mlp_sweep_8500/best/
#
# Runs train_mlp_8500.py in a *new* Python process so Colab never uses a stale in-kernel import.
# If you still see ImportError on "from train_mlp_8500 import ...", DELETE that old cell — use only this one.
#
import os
import subprocess
import sys

REPO = "/content/GNN-Sandia"  # edit if your clone path differs
if os.path.isdir(REPO):
    os.chdir(REPO)
elif not os.path.isfile("train_mlp_8500.py"):
    raise RuntimeError("cd to the repo folder (or set REPO) so train_mlp_8500.py exists.")

if not os.path.isfile("train_mlp_8500.py"):
    raise FileNotFoundError(os.path.abspath("train_mlp_8500.py"))

# Stream child stdout line-by-line — Colab often shows NOTHING with subprocess.run() until the process exits.
_env = {**os.environ, "PYTHONUNBUFFERED": "1"}
_cmd = [sys.executable, "-u", "train_mlp_8500.py", "--epochs", "100", "--batch-size", "16"]
print("Starting (streaming logs):", " ".join(_cmd), flush=True)

proc = subprocess.Popen(
    _cmd,
    cwd=os.getcwd(),
    env=_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f"train_mlp_8500.py exited with code {ret}")

Starting (streaming logs): /usr/bin/python3 -u train_mlp_8500.py --epochs 100 --batch-size 16
[train_mlp_8500] Starting sweep (use: python -u for unbuffered logs in notebooks).
[sweep] Loading tensors from /content/GNN-Sandia/datasets_gnn2/loadtype_8500/dataset_tensors (large files — can take 1–2 min on Colab)...
[sweep] Loaded X,Y shape X=(500, 8541, 14) Y=(500, 8541)  samples=500
[h256_l2] Naive baseline (val) MAE |V| = 0.000896 pu  (test naive MAE = 0.000872 pu)
  arch: hidden_dim=256  num_hidden_layers=2  loss=MSE(|V|) only
  Training (CPU: first epoch can take several minutes — output may look idle).
  epoch   1/100  val_mse_|V|=0.194693  val_mae_|V|_pu=0.422311  val_rmse_|V|_pu=0.441240  lr=1.00e-03
  epoch  10/100  val_mse_|V|=0.029965  val_mae_|V|_pu=0.148543  val_rmse_|V|_pu=0.173104  lr=1.00e-03
  epoch  20/100  val_mse_|V|=0.025370  val_mae_|V|_pu=0.137473  val_rmse_|V|_pu=0.159281  lr=1.00e-03
  epoch  30/100  val_mse_|V|=0.024333  val_mae_|V|_pu=0.134085  val_rmse_|V|_pu=0.155992  lr=5.00e-04
  epoch  40/100  val_mse_|V|=0.024802  val_mae_|V|_pu=0.135087  val_rmse_|V|_pu=0.157487  lr=2.50e-04
  epoch  50/100  val_mse_|V|=0.025086  val_mae_|V|_pu=0.135854  val_rmse_|V|_pu=0.158387  lr=6.25e-05
  epoch  60/100  val_mse_|V|=0.025189  val_mae_|V|_pu=0.136122  val_rmse_|V|_pu=0.158711  lr=1.56e-05
  epoch  70/100  val_mse_|V|=0.025226  val_mae_|V|_pu=0.136220  val_rmse_|V|_pu=0.158828  lr=7.81e-06
  epoch  80/100  val_mse_|V|=0.025239  val_mae_|V|_pu=0.136258  val_rmse_|V|_pu=0.158868  lr=1.95e-06
  epoch  90/100  val_mse_|V|=0.025245  val_mae_|V|_pu=0.136272  val_rmse_|V|_pu=0.158886  lr=1.00e-06
  epoch 100/100  val_mse_|V|=0.025250  val_mae_|V|_pu=0.136285  val_rmse_|V|_pu=0.158902  lr=1.00e-06
[h256_l2] device=cuda  test MAE |V| (pu): 0.147260  vs naive 0.000872  (worse than naive)
  saved /content/GNN-Sandia/datasets_gnn2/loadtype_8500/mlp_sweep_8500/h256_l2/mlp_8500.pt

[h512_l2] Naive baseline (val) MAE |V| = 0.000896 pu  (test naive MAE = 0.000872 pu)
  arch: hidden_dim=512  num_hidden_layers=2  loss=MSE(|V|) only
  Training (CPU: first epoch can take several minutes — output may look idle).
  epoch   1/100  val_mse_|V|=0.189136  val_mae_|V|_pu=0.416480  val_rmse_|V|_pu=0.434898  lr=1.00e-03
  epoch  10/100  val_mse_|V|=0.029836  val_mae_|V|_pu=0.144683  val_rmse_|V|_pu=0.172730  lr=1.00e-03
  epoch  20/100  val_mse_|V|=0.024677  val_mae_|V|_pu=0.133137  val_rmse_|V|_pu=0.157089  lr=5.00e-04
  epoch  30/100  val_mse_|V|=0.024500  val_mae_|V|_pu=0.131465  val_rmse_|V|_pu=0.156524  lr=2.50e-04
  epoch  40/100  val_mse_|V|=0.024721  val_mae_|V|_pu=0.132318  val_rmse_|V|_pu=0.157229  lr=6.25e-05
  epoch  50/100  val_mse_|V|=0.024813  val_mae_|V|_pu=0.132479  val_rmse_|V|_pu=0.157522  lr=1.56e-05
  epoch  60/100  val_mse_|V|=0.024842  val_mae_|V|_pu=0.132551  val_rmse_|V|_pu=0.157613  lr=7.81e-06
  epoch  70/100  val_mse_|V|=0.024857  val_mae_|V|_pu=0.132593  val_rmse_|V|_pu=0.157660  lr=1.95e-06
  epoch  80/100  val_mse_|V|=0.024861  val_mae_|V|_pu=0.132605  val_rmse_|V|_pu=0.157672  lr=1.00e-06
  epoch  90/100  val_mse_|V|=0.024864  val_mae_|V|_pu=0.132612  val_rmse_|V|_pu=0.157684  lr=1.00e-06
  epoch 100/100  val_mse_|V|=0.024870  val_mae_|V|_pu=0.132631  val_rmse_|V|_pu=0.157702  lr=1.00e-06
[h512_l2] device=cuda  test MAE |V| (pu): 0.137678  vs naive 0.000872  (worse than naive)
  saved /content/GNN-Sandia/datasets_gnn2/loadtype_8500/mlp_sweep_8500/h512_l2/mlp_8500.pt

[h768_l2] Naive baseline (val) MAE |V| = 0.000896 pu  (test naive MAE = 0.000872 pu)
  arch: hidden_dim=768  num_hidden_layers=2  loss=MSE(|V|) only
  Training (CPU: first epoch can take several minutes — output may look idle).
  epoch   1/100  val_mse_|V|=0.175445  val_mae_|V|_pu=0.378352  val_rmse_|V|_pu=0.418862  lr=1.00e-03
  epoch  10/100  val_mse_|V|=0.037981  val_mae_|V|_pu=0.159999  val_rmse_|V|_pu=0.194886  lr=1.00e-03
  epoch  20/100  val_mse_|V|=0.021441  val_mae_|V|_pu=0.125213  val_rmse_|V|_pu=0.146427  lr=1.00e-03
  epoch  30/100  val_mse_|V|=0.023440  val_mae_|V|_pu=0.129736  val_rmse_|V|_pu=0.153103  lr=5.00e-04
  epoch  40/100  val_mse_|V|=0.023667  val_mae_|V|_pu=0.131003  val_rmse_|V|_pu=0.153842  lr=1.25e-04
  epoch  50/100  val_mse_|V|=0.023852  val_mae_|V|_pu=0.131601  val_rmse_|V|_pu=0.154442  lr=3.13e-05
  epoch  60/100  val_mse_|V|=0.023925  val_mae_|V|_pu=0.131761  val_rmse_|V|_pu=0.154678  lr=1.56e-05
  epoch  70/100  val_mse_|V|=0.023954  val_mae_|V|_pu=0.131834  val_rmse_|V|_pu=0.154772  lr=3.91e-06
  epoch  80/100  val_mse_|V|=0.023967  val_mae_|V|_pu=0.131880  val_rmse_|V|_pu=0.154813  lr=1.00e-06
  epoch  90/100  val_mse_|V|=0.023976  val_mae_|V|_pu=0.131898  val_rmse_|V|_pu=0.154842  lr=1.00e-06
  epoch 100/100  val_mse_|V|=0.023985  val_mae_|V|_pu=0.131919  val_rmse_|V|_pu=0.154872  lr=1.00e-06
[h768_l2] device=cuda  test MAE |V| (pu): 0.130161  vs naive 0.000872  (worse than naive)
  saved /content/GNN-Sandia/datasets_gnn2/loadtype_8500/mlp_sweep_8500/h768_l2/mlp_8500.pt

[h512_l3] Naive baseline (val) MAE |V| = 0.000896 pu  (test naive MAE = 0.000872 pu)
  arch: hidden_dim=512  num_hidden_layers=3  loss=MSE(|V|) only
  Training (CPU: first epoch can take several minutes — output may look idle).
  epoch   1/100  val_mse_|V|=0.100906  val_mae_|V|_pu=0.277017  val_rmse_|V|_pu=0.317656  lr=1.00e-03
  epoch  10/100  val_mse_|V|=0.029848  val_mae_|V|_pu=0.144286  val_rmse_|V|_pu=0.172767  lr=1.00e-03
  epoch  20/100  val_mse_|V|=0.028737  val_mae_|V|_pu=0.146077  val_rmse_|V|_pu=0.169519  lr=1.00e-03
  epoch  30/100  val_mse_|V|=0.027394  val_mae_|V|_pu=0.141863  val_rmse_|V|_pu=0.165511  lr=5.00e-04
  epoch  40/100  val_mse_|V|=0.027944  val_mae_|V|_pu=0.143599  val_rmse_|V|_pu=0.167165  lr=1.25e-04
  epoch  50/100  val_mse_|V|=0.028129  val_mae_|V|_pu=0.144048  val_rmse_|V|_pu=0.167718  lr=6.25e-05
  epoch  60/100  val_mse_|V|=0.028249  val_mae_|V|_pu=0.144333  val_rmse_|V|_pu=0.168074  lr=1.56e-05
  epoch  70/100  val_mse_|V|=0.028289  val_mae_|V|_pu=0.144428  val_rmse_|V|_pu=0.168192  lr=3.91e-06
  epoch  80/100  val_mse_|V|=0.028302  val_mae_|V|_pu=0.144461  val_rmse_|V|_pu=0.168231  lr=1.95e-06
  epoch  90/100  val_mse_|V|=0.028307  val_mae_|V|_pu=0.144474  val_rmse_|V|_pu=0.168246  lr=1.00e-06
  epoch 100/100  val_mse_|V|=0.028312  val_mae_|V|_pu=0.144487  val_rmse_|V|_pu=0.168262  lr=1.00e-06
[h512_l3] device=cuda  test MAE |V| (pu): 0.147883  vs naive 0.000872  (worse than naive)
  saved /content/GNN-Sandia/datasets_gnn2/loadtype_8500/mlp_sweep_8500/h512_l3/mlp_8500.pt

[h1024_l2] Naive baseline (val) MAE |V| = 0.000896 pu  (test naive MAE = 0.000872 pu)
  arch: hidden_dim=1024  num_hidden_layers=2  loss=MSE(|V|) only
  Training (CPU: first epoch can take several minutes — output may look idle).
  epoch   1/100  val_mse_|V|=0.151984  val_mae_|V|_pu=0.340831  val_rmse_|V|_pu=0.389851  lr=1.00e-03
  epoch  10/100  val_mse_|V|=0.041932  val_mae_|V|_pu=0.169438  val_rmse_|V|_pu=0.204774  lr=1.00e-03
  epoch  20/100  val_mse_|V|=0.030679  val_mae_|V|_pu=0.146481  val_rmse_|V|_pu=0.175154  lr=1.00e-03
  epoch  30/100  val_mse_|V|=0.026945  val_mae_|V|_pu=0.136665  val_rmse_|V|_pu=0.164150  lr=1.00e-03
  epoch  40/100  val_mse_|V|=0.027087  val_mae_|V|_pu=0.139652  val_rmse_|V|_pu=0.164580  lr=5.00e-04
  epoch  50/100  val_mse_|V|=0.027694  val_mae_|V|_pu=0.141119  val_rmse_|V|_pu=0.166415  lr=1.25e-04
  epoch  60/100  val_mse_|V|=0.027914  val_mae_|V|_pu=0.141677  val_rmse_|V|_pu=0.167076  lr=3.13e-05
  epoch  70/100  val_mse_|V|=0.028010  val_mae_|V|_pu=0.141899  val_rmse_|V|_pu=0.167362  lr=1.56e-05
  epoch  80/100  val_mse_|V|=0.028037  val_mae_|V|_pu=0.141962  val_rmse_|V|_pu=0.167441  lr=3.91e-06
  epoch  90/100  val_mse_|V|=0.028050  val_mae_|V|_pu=0.141988  val_rmse_|V|_pu=0.167481  lr=1.00e-06
  epoch 100/100  val_mse_|V|=0.028053  val_mae_|V|_pu=0.141998  val_rmse_|V|_pu=0.167490  lr=1.00e-06
[h1024_l2] device=cuda  test MAE |V| (pu): 0.150931  vs naive 0.000872  (worse than naive)
  saved /content/GNN-Sandia/datasets_gnn2/loadtype_8500/mlp_sweep_8500/h1024_l2/mlp_8500.pt

============================================================
SWEEP DONE. Best architecture: h768_l2
  test MAE |V| (pu) = 0.130161
  copied to /content/GNN-Sandia/datasets_gnn2/loadtype_8500/mlp_sweep_8500/best
  summary: /content/GNN-Sandia/datasets_gnn2/loadtype_8500/mlp_sweep_8500/sweep_summary.json

## Step 7 — Train the GNN baseline (`train_gnn_8500.py`)

Run this after Step 5 so `dataset_tensors/X.pt`, `Y.pt`, `Y_angle.pt` exist, and after Step 4 so `graph_tensors/edge_index.pt` and `edge_attr.pt` exist.

This is a first-pass GCN-style residual model that predicts `(vmag_pu, vang_deg)` when `Y_angle.pt` is available; otherwise it trains on `vmag_pu` only.

In [ ]:
# Step 7 — GNN training (fresh subprocess to avoid stale imports)
import os
import subprocess
import sys

REPO = "/content/GNN-Sandia"  # Colab path
if os.path.isdir(REPO):
    os.chdir(REPO)

if not os.path.isfile("train_gnn_8500.py"):
    raise FileNotFoundError(os.path.abspath("train_gnn_8500.py"))

_env = {**os.environ, "PYTHONUNBUFFERED": "1"}

_cmd = [
    sys.executable,
    "-u",
    "train_gnn_8500.py",
    "--epochs",
    "5",
    "--hidden-dim",
    "64",
    "--num-layers",
    "3",
    "--max-train-samples",
    "20",
    "--max-val-samples",
    "20",
    "--max-test-samples",
    "10",
]

print("Starting GNN training:", " ".join(_cmd), flush=True)

proc = subprocess.Popen(
    _cmd,
    cwd=os.getcwd(),
    env=_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f"train_gnn_8500.py exited with code {ret}")

Starting GNN training: /usr/bin/python3 -u train_gnn_8500.py --epochs 5 --hidden-dim 64 --num-layers 3 --max-train-samples 20 --max-val-samples 20 --max-test-samples 10
[train_gnn_8500] device=cuda  S=500 N=8541 F=14 angle_target=OFF
  epoch   1/5  train_loss=0.062109  val_mae_|V|=0.090193
  epoch   2/5  train_loss=0.013216  val_mae_|V|=0.063377
  epoch   3/5  train_loss=0.010626  val_mae_|V|=0.061782
  epoch   4/5  train_loss=0.007023  val_mae_|V|=0.053858
  epoch   5/5  train_loss=0.005983  val_mae_|V|=0.047271
[train_gnn_8500] done. out_dir=/content/GNN-Sandia/datasets_gnn2/loadtype_8500/gnn_8500_baseline/res_gcn_h64_L3
  test MAE |V| (pu): 0.042507

## Step 8 — Daily OpenDSS vs MLP vs GNN comparison (`compare_daily_8500_mlp_gnn.py`)

Runs a 24h (288-step) 8500-node daily simulation and compares voltage magnitude trajectories for chosen nodes.

- Inputs: **two checkpoint paths** (`gnn_8500.pt`, `mlp_8500.pt`)
- Output: per-node plots (`OpenDSS vs MLP vs GNN`) and timing summary
- This script uses the optimized inference path (pinned host buffers, 2 CUDA streams, vectorized feature build).

In [4]:
# Step 8 — run daily OpenDSS vs MLP vs GNN (streaming logs)
import os
import subprocess
import sys

# Auto-detect repo root so this cell works both locally and in Colab.
SCRIPT_NAME = "compare_daily_8500_mlp_gnn.py"
CANDIDATES = [
    os.getcwd(),
    r"C:\Users\alita\OneDrive\Desktop\GNN2",  # local Windows fallback
    "/content/GNN-Sandia",                        # Colab fallback
]

repo = None
for p in CANDIDATES:
    if os.path.isfile(os.path.join(p, SCRIPT_NAME)):
        repo = p
        break

if repo is None:
    raise FileNotFoundError(
        f"Could not find {SCRIPT_NAME}. Tried: {CANDIDATES}. "
        "Set CANDIDATES or cd to repo root first."
    )

os.chdir(repo)
print("Using repo:", os.getcwd())

# TODO: set these two checkpoint paths
GNN_CKPT = "datasets_gnn2/loadtype_8500/gnn_8500_baseline/res_gcn_h64_L3/gnn_8500.pt"
MLP_CKPT = "datasets_gnn2/loadtype_8500/mlp_sweep_8500/best/mlp_8500.pt"

# Nodes to plot.
# IMPORTANT: 8500-node IDs are names like m1026891.1, l3216367.2, sourcebus.1, ...
# (not 34-node IDs such as 816.1 / 840.1 / 848.2).
PLOT_NODES = ["m1026891.1", "m1026891.2", "m1026891.3"]

# Validate/fix node list against the 8500 node index.
import pandas as pd
_node_csv = os.path.join("datasets_gnn2", "loadtype_8500", "gnn_node_index_master.csv")
if not os.path.isfile(_node_csv):
    raise FileNotFoundError(os.path.abspath(_node_csv))
_all_nodes = set(pd.read_csv(_node_csv)["node"].astype(str).tolist())
_valid_nodes = [n for n in PLOT_NODES if n in _all_nodes]
if len(_valid_nodes) == 0:
    # Fallback to first few non-source nodes if user-provided list is invalid.
    _df_nodes = pd.read_csv(_node_csv)
    _fallback = [n for n in _df_nodes["node"].astype(str).tolist() if not n.lower().startswith("sourcebus")][:3]
    _valid_nodes = _fallback
    print("[WARN] No requested plot nodes found in 8500 index. Falling back to:", _valid_nodes)
elif len(_valid_nodes) < len(PLOT_NODES):
    _missing = [n for n in PLOT_NODES if n not in _all_nodes]
    print("[WARN] Dropping nodes not in 8500 index:", _missing)

_env = {**os.environ, "PYTHONUNBUFFERED": "1"}
_cmd = [
    sys.executable,
    "-u",
    SCRIPT_NAME,
    GNN_CKPT,
    MLP_CKPT,
    "--nodes",
    *_valid_nodes,
    "--output-dir",
    "gnn2_daily_compare_8500_output",
]

print("Starting daily 8500 compare:", " ".join(_cmd), flush=True)

proc = subprocess.Popen(
    _cmd,
    cwd=os.getcwd(),
    env=_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f"{SCRIPT_NAME} exited with code {ret}")

Using repo: C:\Users\alita\OneDrive\Desktop\GNN2
Starting daily 8500 compare: c:\Users\alita\OneDrive\Desktop\quest-ssim\env\Scripts\python.exe -u compare_daily_8500_mlp_gnn.py datasets_gnn2/loadtype_8500/gnn_8500_baseline/res_gcn_h64_L3/gnn_8500.pt datasets_gnn2/loadtype_8500/mlp_sweep_8500/best/mlp_8500.pt --nodes m1026891.1 m1026891.2 m1026891.3 --output-dir gnn2_daily_compare_8500_output
C:\Users\alita\OneDrive\Desktop\GNN2\compare_daily_8500_mlp_gnn.py:113: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer b

RuntimeError: compare_daily_8500_mlp_gnn.py exited with code 1